# Epsilon Fund — Strategy Testing
---

In [5]:
import pandas as pd
import numpy as np
import sys

# ── Set your repo root path ────────────────────────────────────────────────────
ROOT = r'/Users/justiniturregui/Desktop/epsilon/github/Epsilon-Quant-Research'  # ← change to yours
# ──────────────────────────────────────────────────────────────────────────────

sys.path.append(ROOT + '/infrastructure/data')
sys.path.append(ROOT + '/infrastructure/backtester')

from binance_client import get_binance_client, get_data, get_multiple_data
from engine import backtest

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: `365` (1y) · `730` (2y) · `1825` (5y) · `2555` (7y, recommended minimum)

In [6]:
SYMBOL   = 'BTCUSDT'
INTERVAL = '1d'
LOOKBACK = 2555

# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client = get_binance_client()
df = get_data(client, SYMBOL, INTERVAL, LOOKBACK)

print(f'{SYMBOL} | {INTERVAL} | {df.index[0].date()} -> {df.index[-1].date()} | {len(df)} rows')
df.tail(3)

BTCUSDT | 1d | 2019-03-08 -> 2026-03-05 | 2555 rows


,Open,High,Low,Close,Volume
Time,,,,,
2026-03-03,68830.06,69258.08,66158.00,68338.00,24972.24180
2026-03-04,68338.01,74050.00,67400.00,72666.77,44919.07610
2026-03-05,72666.77,73558.15,70645.47,71316.66,24175.01596


---
## Strategy

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> Signals are shifted 1 bar by the engine — no need to shift manually.

In [7]:
strategy_df = df.copy()

# ── Strategy logic ─────────────────────────────────────────────────────────────

# --- ATR (Wilder's smoothing, matches Pine Script RMA) ---
def calculate_atr(df, period):
    high       = df['high']
    low        = df['low']
    prev_close = df['close'].shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

# --- SUPERTREND ---
atr_period = 10
factor     = 3.0

atr = calculate_atr(strategy_df, atr_period)
hl2 = (strategy_df['high'] + strategy_df['low']) / 2

raw_upper = hl2 + factor * atr
raw_lower = hl2 - factor * atr

n          = len(strategy_df)
upper      = np.zeros(n)
lower      = np.zeros(n)
direction  = np.zeros(n)   # -1 = bullish, +1 = bearish

for i in range(n):
    if i == 0:
        upper[i]     = raw_upper.iloc[i]
        lower[i]     = raw_lower.iloc[i]
        direction[i] = 1
        continue

    close_prev = strategy_df['close'].iloc[i - 1]
    close_curr = strategy_df['close'].iloc[i]

    upper[i] = raw_upper.iloc[i] if (raw_upper.iloc[i] < upper[i-1] or close_prev > upper[i-1]) \
               else upper[i-1]
    lower[i] = raw_lower.iloc[i] if (raw_lower.iloc[i] > lower[i-1] or close_prev < lower[i-1]) \
               else lower[i-1]

    if direction[i-1] == 1:
        direction[i] = -1 if close_curr > upper[i-1] else 1
    else:
        direction[i] =  1 if close_curr < lower[i-1] else -1

strategy_df['supertrend'] = np.where(direction == -1,
                                     lower,   # bullish: lower band is the support line
                                     upper)   # bearish: upper band is the resistance line
strategy_df['direction']  = direction

# --- PULLBACK ZONE ---
atr_len      = 10
pullback_atr = calculate_atr(strategy_df, atr_len)

strategy_df['long_in_range'] = (
    (strategy_df['direction'] == -1) &
    (strategy_df['close'] >= strategy_df['supertrend']) &
    (strategy_df['close'] <= strategy_df['supertrend'] + pullback_atr)
)
strategy_df['short_in_range'] = (
    (strategy_df['direction'] == 1) &
    (strategy_df['close'] <= strategy_df['supertrend']) &
    (strategy_df['close'] >= strategy_df['supertrend'] - pullback_atr)
)

# --- ENTRY TRIGGERS ---
strategy_df['long_trigger']  = (strategy_df['high'].shift(1) + strategy_df['high'])  / 2
strategy_df['short_trigger'] = (strategy_df['low'].shift(1)  + strategy_df['low'])   / 2

# --- EXIT LEVELS ---
strategy_df['long_exit_level']  = (strategy_df['low'].shift(1)  + strategy_df['low'].shift(2))  / 2
strategy_df['short_exit_level'] = (strategy_df['high'].shift(1) + strategy_df['high'].shift(2)) / 2

# --- POSITION LOOP ---
strategy_df['position'] = 0
in_pos = 0

for i in range(2, len(strategy_df)):   # start at 2: exit levels need 2 prior bars
    idx      = strategy_df.index[i]
    curr     = strategy_df.iloc[i]
    prev     = strategy_df.iloc[i - 1]

    # Skip warmup bars where ATR/supertrend not yet valid
    if pd.isna(curr['supertrend']) or pd.isna(curr['long_trigger']):
        continue

    if in_pos == 1:   # in long
        defensive     = curr['close'] < curr['supertrend']
        trailing_exit = curr['close'] < curr['long_exit_level']
        if defensive or trailing_exit:
            in_pos = 0

    elif in_pos == -1:   # in short
        defensive     = curr['close'] > curr['supertrend']
        trailing_exit = curr['close'] > curr['short_exit_level']
        if defensive or trailing_exit:
            in_pos = 0

    if in_pos == 0:   # flat — check entries
        if prev['long_in_range'] and curr['close'] > curr['long_trigger']:
            in_pos = 1
        elif prev['short_in_range'] and curr['close'] < curr['short_trigger']:
            in_pos = -1

    strategy_df.loc[idx, 'position'] = in_pos

# ──────────────────────────────────────────────────────────────────────────────

strategy_df = strategy_df.dropna(subset=['supertrend', 'long_exit_level'])
strategy_df['position'] = strategy_df['position'].fillna(0).astype(int)

strategy_df['position'].value_counts()

KeyError: 'high'

---
## Backtest

| Parameter | Options | Default |
|---|---|---|
| `cost` | Cost per trade as decimal — `0.001` = 0.1% | `0.0` |
| `show_plot` | `True` / `False` | `True` |
| `save_html` | Filename string or `None` | `None` |
| `show_trades` | Overlay entry/exit markers on price chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | same asset |

In [ ]:
results = backtest(
    data           = strategy_df,
    cost           = 0.0,
    show_plot      = True,
    save_html      = None,
    show_trades    = False,
    benchmark_data = None
)

print(f"Return        {results['total_return']*100:>8.2f}%")
print(f"Sharpe        {results['sharpe_ratio']:>8.2f}")
print(f"Max Drawdown  {results['max_drawdown']*100:>8.2f}%")
print(f"Profit Factor {results['profit_factor']:>8.2f}")
print(f"Win Rate      {results['win_rate']*100:>8.2f}%")
print(f"# Trades      {results['num_trades']:>8}")

ValueError: Data must have 'Close' column